# AnionXAS end-to-end eight-element training

Native clean Figshare AnionXAS targets use 200 points for Ti, V, Cr, Mn, Fe, Co, Ni, and Cu. Structures are read from `$OMNIXAS_DATA_ROOT/anionxas_curated_200/extracted/FEFF`. The workflow uses the package global material split, skips and reports missing structures, exports scaled 64D encoder features, then trains UniversalXAS and validation-selected tuned heads.


In [ ]:
from pathlib import Path
import os
# Work from either the repository root or tutorial_omnixas/.
HERE = Path.cwd().resolve()
REPO = HERE if (HERE / "tutorial_omnixas").is_dir() else HERE.parent
TUTORIAL = REPO / "tutorial_omnixas"
DATA_ROOT = Path(os.environ.get("OMNIXAS_DATA_ROOT", REPO.parent / "OmniXAS_data"))
TARGET = TUTORIAL / "anionxas_targets_200.npz"
RAW = DATA_ROOT / "anionxas_curated_200" / "extracted" / "FEFF"
OUT = REPO / "output" / "anionxas_e2e_8elem"
SCRIPT = TUTORIAL / "train_anionxas_e2e_8elem.py"
print(TARGET, RAW, OUT)


## Preflight
This checks the native package, global `split_codes`, eight-element rows, and structure availability without starting MatGL training.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, str(SCRIPT), "--target-package", str(TARGET),
                "--data-root", str(DATA_ROOT), "--preflight"], check=True)


## Full training
The command below is intentionally not run while building this notebook. Checkpoint selection uses validation only; test metrics are written once at the end. Adjust `--gpu`, worker count, or epochs for the machine.

In [ ]:
# Full training is opt-in because it can require substantial GPU time.
RUN_TRAINING = False
TRAINING_COMMAND = [sys.executable, str(SCRIPT),
                    "--target-package", str(TARGET), "--data-root", str(DATA_ROOT),
                    "--output-root", str(OUT), "--gpu", "0", "--num-workers", "4"]
if RUN_TRAINING:
    subprocess.run(TRAINING_COMMAND, check=True)


In [ ]:
import pandas as pd
metrics = pd.read_csv(OUT / "seed42" / "metrics.csv")
metrics


In [ ]:
import matplotlib.pyplot as plt
val = metrics[metrics.split == "val"]
for variant, group in val.groupby("variant"):
    plt.plot(group.element, group.eta, "o-", label=variant)
plt.ylabel("validation eta (higher is better)"); plt.xlabel("absorbing element"); plt.legend(); plt.grid(alpha=.25); plt.show()


In [ ]:
import json
provenance = json.loads((OUT / "seed42" / "provenance.json").read_text())
display(provenance)
